In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("Data Cleaning Curated 2")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .getOrCreate()
)
# Silence any WARN-level logs from this point forth.
spark.sparkContext.setLogLevel("ERROR")

Lets read in our cleaned rental data

In [5]:
curated_path = "../../rea_data/curated/vic_rentals_curated_final.parquet"

df_curated = spark.read.parquet(curated_path)

<br>
<br>
<br>

### Merging With Crime Rate Data

Crime rate data is granularised by Local Government Area (LGA), so in order to merge with our curated rental data, we must use the `lat` and `lon` information to calculate which LGA a given listing lies within. Then merging on LGA becomes very simple.

##### How we did it:
In the following cell we used the ABS LGA shapefile which gives each Victorian council's boundary as a polygon. With geopandas, we turned each property's latitude and longitude into a point and converted it to the shapefile's coordinate system. A spatial join (sjoin with predicate="within") then found which LGA polygon each point falls inside. Finally, we removed the ABS " (Vic.)" suffix from the names so they match the Crime Statistics Agency's LGA names, and stored the result in a new LGA column.

In [7]:
from pathlib import Path

import geopandas as gpd

LGA_FILE = Path("../../external_data/LGA_2025_AUST_GDA2020/LGA_2025_AUST_GDA2020.shp")
ABS_LGA_COL = "LGA_NAME25"

# Load LGA boundaries
lgas = gpd.read_file(LGA_FILE)

# Spark -> pandas, with lat/lon as numbers
df_pd = df_curated.toPandas()
df_pd["lat"] = df_pd["lat"].astype(float)
df_pd["lon"] = df_pd["lon"].astype(float)

# Turn each property into a point
properties = gpd.GeoDataFrame(
    df_pd,
    geometry=gpd.points_from_xy(df_pd["lon"], df_pd["lat"]),
    crs="EPSG:4326",
)
properties = properties.to_crs(lgas.crs)

# Find which LGA each property falls inside
properties = gpd.sjoin(
    properties,
    lgas[[ABS_LGA_COL, "geometry"]],
    how="left",
    predicate="within",
)

# Clean LGA names to match the CSA format, e.g. "Kingston (Vic.)"
properties["LGA"] = (
    properties[ABS_LGA_COL]
    .str.replace(r"\s*\(Vic\.\)", "", regex=True)
    .str.strip()
)
properties = properties.drop(
    columns=["geometry", "index_right", ABS_LGA_COL]
)

print(properties["LGA"].isna().sum(), "properties with no LGA")

# Back to Spark
df_curated = spark.createDataFrame(properties)
df_curated.show(1, vertical=True)

0 properties with no LGA
-RECORD 0--------------------------------------------------
 lat                                | -37.830982           
 lon                                | 144.87091            
 listing_id                         | 16782629             
 suburb                             | SOUTH KINGSVILLE     
 weekly_rent                        | 460.0                
 date_listed                        | 2025-08-13           
 bedrooms                           | 2.0                  
 bathrooms                          | 1.0                  
 carspaces                          | 2.0                  
 property_type                      | Apartment / Unit ... 
 feat_built_in_wardrobes            | 1                    
 feat_heating                       | 1                    
 feat_dishwasher                    | 0                    
 feat_secure_parking                | 1                    
 feat_internal_laundry              | 0                    
 feat_air_condi

<br>
<br>
<br>

### Merging on LGA with the Crime rate data 

Firstly, read in our cleaned crime rate data.

In [8]:
CRIME_FILE = "../../external_data/vic_crime_by_lga.csv"

crime = spark.read.csv(CRIME_FILE, header=True, inferSchema=True)

crime.printSchema()
crime.show(5)

root
 |-- Local Government Area: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Rate per 100,000 population: double (nullable = true)

+---------------------+----+---------------------------+
|Local Government Area|Year|Rate per 100,000 population|
+---------------------+----+---------------------------+
|               Alpine|2012|           3840.11891981171|
|               Ararat|2012|          9977.944419938243|
|             Ballarat|2012|         10283.179450147009|
|              Banyule|2012|          5829.288025889968|
|           Bass Coast|2012|          6369.218709171674|
+---------------------+----+---------------------------+
only showing top 5 rows


Before we merge, lets ensure we change `Colac Otway` to `Colac-Otway` within our rental data (which its how it was represented in the crime rate files). This is something we noticed through a physical check.

In [9]:
# Fix the Colac Otway spelling to match the CSA
df_curated = df_curated.withColumn(
    "LGA",
    F.when(F.col("LGA") == "Colac Otway", "Colac-Otway")
    .otherwise(F.col("LGA")),
)


Now, lets perform the merge step and check for any rows that don't end up with a crime rate.

In [10]:
# Match the crime data's column names to df_curated
crime = (
    crime
    .withColumnRenamed("Local Government Area", "LGA")
    .withColumnRenamed("Rate per 100,000 population", "crime_rate")
    .select("Year", "LGA", "crime_rate")
)

# Crime year for each property (CSA years end in June)
listed = F.to_date(F.col("date_listed"))
df_curated = df_curated.withColumn(
    "Year",
    F.year(listed) + (F.month(listed) > 6).cast("int"),
)

# Join on LGA + Year
rows_before = df_curated.count()
df_curated = df_curated.join(crime, on=["LGA", "Year"], how="left")
rows_after = df_curated.count()

print(rows_before, "rows before,", rows_after, "rows after")
missing = df_curated.filter(F.col("crime_rate").isNull()).count()
print(missing, "properties with no crime rate")

12317 rows before, 12317 rows after
1 properties with no crime rate


In [11]:
no_rate = df_curated.filter(F.col("crime_rate").isNull())

no_rate.select("date_listed", "Year", "LGA", "lat", "lon").orderBy(
    "Year"
).show(1, truncate=False)

+-----------+----+------------------+----------+---------+
|date_listed|Year|LGA               |lat       |lon      |
+-----------+----+------------------+----------+---------+
|2025-08-28 |2026|Unincorporated Vic|-37.146248|146.45332|
+-----------+----+------------------+----------+---------+



Finally, we will drop this particular row since it is one row of $\approx 12,300$ and an `LGA` = "Unincorporated Vic" doesn't actually have a crime rate.

In [12]:
df_curated = df_curated.filter(F.col("crime_rate").isNotNull())

In [13]:
no_rate = df_curated.filter(F.col("crime_rate").isNull())

no_rate.select("date_listed", "Year", "LGA", "lat", "lon").orderBy(
    "Year"
).show(1, truncate=False)

+-----------+----+---+---+---+
|date_listed|Year|LGA|lat|lon|
+-----------+----+---+---+---+
+-----------+----+---+---+---+



Create a table showwing the average crime rate per year to undertand the data better.

In [14]:
from pyspark.sql import functions as F

(
    df_curated.groupBy("Year")
    .agg(
        F.round(F.avg("crime_rate"), 1).alias("avg_crime_rate"),
        F.count("*").alias("listings"),
    )
    .orderBy("Year")
    .show()
)

+----+--------------+--------+
|Year|avg_crime_rate|listings|
+----+--------------+--------+
|2012|        4421.0|       1|
|2013|       10955.3|      10|
|2014|       11497.4|      12|
|2015|        4238.0|       1|
|2016|       20505.9|       3|
|2017|        7768.5|       5|
|2018|        9329.3|       1|
|2019|       12738.2|       8|
|2020|        9566.3|       9|
|2021|        6937.2|       7|
|2022|        6979.2|      24|
|2023|        8651.2|      51|
|2024|        9625.8|     103|
|2025|       10151.5|    1151|
|2026|       10008.3|   10930|
+----+--------------+--------+



Now, lets finally save this first merged data set to disc.

In [16]:
OUTPUT_PATH = "../../rea_data/analysis/vic_rentals_merged_1.parquet"

df_curated.write.mode("overwrite").parquet(OUTPUT_PATH)
print("Saved to", OUTPUT_PATH)

Saved to ../../rea_data/analysis/vic_rentals_merged_1.parquet
